# YOLOv8 Crab Detector — Training on Google Colab

> **Prima di iniziare:** vai su `Runtime → Change runtime type → T4 GPU`

## 1. Installa Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics

# Verifica GPU disponibile
import torch
print("GPU disponibile:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Carica e decomprimi il dataset

Carica il file `dataset.zip` generato in precedenza.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # seleziona dataset.zip

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/")

print("Contenuto /content/dataset:")
for split in ("images/train", "images/val", "labels/train", "labels/val"):
    path = f"/content/dataset/{split}"
    n = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"  {split}: {n} file")

## 3. Aggiorna data.yaml

Il path assoluto nel `data.yaml` va aggiornato alla posizione Colab.

In [ ]:
import yaml

yaml_path = "/content/dataset/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data["path"] = "/content/dataset"

with open(yaml_path, "w") as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print("data.yaml aggiornato:")
print(yaml.dump(data))

## 4. Training

Allena YOLOv8n sul dataset. Cambia `yolov8n.pt` con `yolov8s.pt` per un modello più preciso (ma più lento).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # scarica automaticamente i pesi pre-trainati

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="crabs_yolo",
    project="/content/runs",
    exist_ok=True,
    patience=15,        # early stopping se non migliora per 15 epoche
    device=0,           # usa GPU
)

## 5. Valutazione sul validation set

In [ ]:
best_model = YOLO("/content/runs/crabs_yolo/weights/best.pt")
metrics = best_model.val(data="/content/dataset/data.yaml")

print(f"\nmAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

## 6. Preview predizioni su immagini di validazione

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random
import os

val_dir = "/content/dataset/images/val"
val_imgs = random.sample(os.listdir(val_dir), 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, fname in zip(axes, val_imgs):
    img_path = os.path.join(val_dir, fname)
    result = best_model(img_path, verbose=False)[0]
    annotated = result.plot()  # BGR con box disegnati
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.axis("off")
    ax.set_title(fname, fontsize=8)

plt.tight_layout()
plt.show()

## 7. Scarica il modello trainato (`best.pt`)

In [ ]:
from google.colab import files
files.download("/content/runs/crabs_yolo/weights/best.pt")